# BekaaSense — Exploratory Data Analysis

Run this notebook after you have a populated `data/processed/bekaa_valley_clean.csv`.

From the repo root:

```bash
python scripts/generate_synthetic.py   # demo data
# or:
python scripts/build_canonical.py      # real data from data/raw/
```

This notebook reproduces the key EDA findings:

1. Precipitation is the master driver (r ≈ 0.98 with De Martonne).
2. Monthly-scale aridity is dominated by Hyper-arid classes (class imbalance).
3. Station-level divergence: Ammik improving, others declining.
4. Mann-Kendall trends are not yet statistically significant (p > 0.05).
5. Seasonal memory is short (dm_lag1 > dm_lag2 > dm_lag3).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

from data_ingestion.loaders import load_clean_csv
from data_ingestion.features import build_features

df = load_clean_csv('../data/processed/bekaa_valley_clean.csv')
feat = build_features(df)
print(f'Shape: {feat.shape}')
feat.head()

## 1. Correlation with the target

In [ ]:
corr_cols = ['precip_sum', 'temp_avg', 'temp_max', 'temp_min',
             'precip_roll3', 'precip_roll6', 'dm_lag1', 'dm_lag2',
             'dm_lag3', 'dm_roll3', 'spi3']
corr = feat[['de_martonne'] + corr_cols].corr()['de_martonne'].sort_values()
corr.drop('de_martonne').plot.barh(figsize=(7, 5), color='steelblue')
plt.axvline(0, color='black', linewidth=0.5)
plt.title('Correlation with De Martonne index')
plt.tight_layout()
plt.show()

## 2. Aridity class imbalance per station

In [ ]:
counts = (feat.groupby(['station', 'aridity_zone']).size()
              .unstack(fill_value=0))
order = ['Hyper-arid', 'Arid', 'Semi-arid', 'Sub-humid', 'Humid']
counts = counts[[c for c in order if c in counts.columns]]
counts.plot.bar(stacked=True, figsize=(9, 5),
                color=['#dc2626', '#f97316', '#eab308', '#22c55e', '#2563eb'])
plt.title('Aridity zone distribution by station')
plt.ylabel('Months')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 3. Station time-series of De Martonne

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for station, g in feat.groupby('station'):
    g = g.sort_values(['year', 'month'])
    dates = pd.to_datetime(dict(year=g.year, month=g.month, day=1))
    ax.plot(dates, g['de_martonne'], label=station, alpha=0.8)
for z_name, lo, hi in [('Hyper-arid', 0, 5), ('Arid', 5, 10),
                       ('Semi-arid', 10, 20), ('Sub-humid', 20, 30)]:
    ax.axhline(hi, color='grey', linewidth=0.3, linestyle=':')
ax.legend()
ax.set_title('Monthly De Martonne index per station')
ax.set_ylabel('De Martonne')
plt.tight_layout()
plt.show()

## 4. Mann-Kendall trend test (honest limitation)

All p-values are expected to be > 0.05 with only ~10 years of annual data.

In [ ]:
from scipy.stats import kendalltau

rows = []
for station, g in feat.groupby('station'):
    annual = g.groupby('year')['de_martonne'].mean()
    if len(annual) < 4:
        continue
    tau, pval = kendalltau(annual.index, annual.values)
    slope = np.polyfit(annual.index, annual.values, 1)[0]
    rows.append({'station': station,
                 'direction': 'INCREASING' if slope > 0 else 'DECLINING',
                 'slope_DM_per_yr': slope,
                 'p_value': pval, 'tau': tau})
pd.DataFrame(rows).round(3)